In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#Pasta do Drive com o modelo do BERT: https://drive.google.com/drive/folders/1-O2rYPftWIiU31xqfWWWQxUScFv1O88N?usp=sharing

Mounted at /content/drive


In [ ]:
BASE_PATH = "/content/drive/MyDrive/models_ner"

In [ ]:
import os
os.makedirs(BASE_PATH, exist_ok=True)

In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer

BERT_PATH = "/content/drive/MyDrive/models_ner/bert_ner"

bert_token_classifier_model = AutoModelForTokenClassification.from_pretrained(BERT_PATH)
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_PATH)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
import joblib

ngram_data = joblib.load("/content/drive/MyDrive/models_ner/ngram_model.pkl")

ngram_vectorizer = ngram_data["vectorizer"]
ngram_model = ngram_data["model"]

### Test BERT Token Classification Model

This section will use the loaded BERT model (`model` from `transformers`) and its tokenizer to perform Named Entity Recognition (NER) on a sample sentence.

In [ ]:
import torch

generic_text = "Google was founded by Larry Page and Sergey Brin while they were Ph.D. students at Stanford University in California."
#generic_text = "Bertrand Lira mora na cidade de Joâo Pessoa e estudou na Universidade Federal da Paraíba. Aos 25 anos de idade, ganhou o Prêmio Turing por um software que fazia Reconhecimento de Entidades Nomeadas usando um fine tuning com o modelo BERT. Após isso, foi convidado a ser membro executivo do Google pelo seu grande feito."

inputs = bert_tokenizer(generic_text, return_tensors="pt", truncation=True)

with torch.no_grad():
    outputs = bert_token_classifier_model(**inputs)

predictions = torch.argmax(outputs.logits, dim=2)

tokens = bert_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
labels = [bert_token_classifier_model.config.id2label[p.item()] for p in predictions[0]]

print("BERT Token Classification Results (Reassembled Words):")

current_word = ""
current_label = "O"

for token, label in zip(tokens, labels):
    if token.startswith("##"):
        current_word += token[2:]
    elif token in ['[CLS]', '[SEP]', '[PAD]', '[UNK]']:
        if current_word:
            print(f"{current_word} ({current_label})")
        current_word = ""
        current_label = "O"

    else:
        if current_word:
            print(f"{current_word} ({current_label})")
        current_word = token
        current_label = label

if current_word:
    print(f"{current_word} | ({current_label})")

BERT Token Classification Results (Reassembled Words):
Bertrand (B-PER)
Lira (I-PER)
mora (O)
na (O)
cidade (O)
de (O)
Joâo (B-LOC)
Pessoa (I-LOC)
e (O)
estudou (O)
na (O)
Universidade (B-LOC)
Federal (I-LOC)
da (I-LOC)
Paraíba (I-LOC)
. (O)
Aos (O)
25 (O)
anos (O)
de (O)
idade (O)
, (O)
ganhou (O)
o (O)
Prêmio (O)
Turing (O)
por (O)
um (O)
software (O)
que (O)
fazia (O)
Reconhecimento (O)
de (O)
Entidades (O)
Nomeadas (O)
usando (O)
um (O)
fine (O)
tuning (O)
com (O)
o (O)
modelo (O)
BERT (O)
. (O)
Após (O)
isso (O)
, (O)
foi (O)
convidado (O)
a (O)
ser (O)
membro (O)
executivo (O)
do (O)
Google (B-ORG)
pelo (O)
seu (O)
grande (O)
feito (O)
. (O)
